In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from pathlib import Path

In [2]:
DATA_DIR = Path("/Users/melodyqian/Documents/GitHub/FindMyNuclearWaste/data/shapefile")  # folder with tl_2024_XX_bg.zip files
N_POINTS = 70
OUTPUT_CSV = "random_bg_points_2024.csv"

In [3]:
zips = sorted(DATA_DIR.glob("tl_2024_*_bg.zip"))
if not zips:
    raise FileNotFoundError(f"No 'tl_2024_XX_bg.zip' files found in {DATA_DIR}")

In [4]:
# Read and concat all states (GeoPandas can read straight from .zip paths)
parts = []
for z in zips:
    # Each zip contains a shapefile; gpd.read_file can point at the zip directly
    parts.append(gpd.read_file(f"zip://{z}"))
bg = pd.concat(parts, ignore_index=True)
bg = gpd.GeoDataFrame(bg, geometry="geometry", crs=bg.crs)

# Drop empties just in case
bg = bg[~bg.geometry.is_empty & bg.geometry.notnull()].copy()

In [5]:
# Project to an equal-area CRS for correct area weights (covers all US states)
# EPSG:2163 = US National Atlas Equal Area
bg_eq = bg.to_crs(2163)
areas = bg_eq.geometry.area.values

# Build sampling probabilities (uniform over land area)
weights = np.where(np.isfinite(areas) & (areas > 0), areas, 0)
if weights.sum() == 0:
    raise ValueError("Areas are zero/invalid after reprojection; check data.")
probs = weights / weights.sum()

In [6]:
# Work in WGS84 for output
bg = bg.to_crs(4326)
geoms = bg.geometry.values

rng = np.random.default_rng(12345)
# fixed seed for reproducibility

In [9]:

def random_point_in_poly(poly):
    """Rejection sample uniformly inside poly via bbox hits."""
    minx, miny, maxx, maxy = poly.bounds
    for _ in range(2000):
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(x, y)
        if poly.contains(p):
            return p
    # Fallback (rare): a guaranteed-inside point
    return poly.representative_point()

# Draw polygons by area-weight and sample a point in each
idxs = rng.choice(len(geoms), size=N_POINTS, p=probs, replace=True)
points = [random_point_in_poly(geoms[i]) for i in idxs]

In [ ]:
# Prepare output
lats = [p.y for p in points]
lons = [p.x for p in points]
df = pd.DataFrame({"latitude": lats, "longitude": lons})

# Print plain "lat, lon" lines
for lat, lon in zip(lats, lons):
    print(f"{lat:.6f}, {lon:.6f}")

# Save CSV
df.to_csv(OUTPUT_CSV, index=False)


63.714453, -158.890828
38.815899, -120.839732
35.857179, -107.253691
48.239396, -104.433209
38.328346, -107.792141
40.938970, -122.695430
45.795060, -95.840243
62.746212, -156.079282
47.750621, -113.886290
43.022222, -96.706296
36.352787, -113.926199
44.005062, -97.157139
48.095755, -112.435326
69.124630, -155.575621
39.799683, -89.932479
42.169505, -117.173676
45.606488, -112.365188
36.017951, -119.146275
42.737780, -102.987594
64.209190, -152.154690
68.004469, -146.170296
60.212326, -154.853823
38.658646, -120.288704
41.589172, -92.174569
36.389960, -111.332090
35.322073, -106.319360
67.755906, -143.305644
65.368723, -141.513872
69.497343, -141.436659
45.530058, -94.036689
40.213635, -82.880929
45.763652, -94.639009
45.151428, -99.554866
40.735379, -100.157089
36.457279, -95.702033
43.999385, -102.829476
44.792509, -67.926986
44.934649, -102.202528
39.934350, -95.516247
35.272031, -112.177627
39.358830, -87.052690
47.887712, -114.142696
36.051456, -117.209328
42.410924, -122.291699
3